# Step 4 — Define the Action Space and Validity Rules

This notebook completes **Step 4 only**.

Goal: define a small explicit action set, build deterministic state-dependent validity rules, and export `valid_action_space.csv`.

## What we do in this step (simple view)

1. Load `case_step_features` from Step 2.
2. Define a manager-level action set and integer IDs.
3. Build deterministic validity rules from risk, delay, workload, and volume proxies.
4. Evaluate masks over all states.
5. Run checks and export `valid_action_space.csv`.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

OUTPUT_DIR = Path('./output')
FEATURES_PARQUET = OUTPUT_DIR / 'case_step_features.parquet'
FEATURES_CSV = OUTPUT_DIR / 'case_step_features.csv'
ACTION_SPACE_PATH = OUTPUT_DIR / 'valid_action_space.csv'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Output dir:', OUTPUT_DIR.resolve())

Output dir: /home/praharsha/Desktop/bueracratic-workflow-analyzer/output


In [2]:
if not FEATURES_PARQUET.exists():
    print('Error: previous notebook should be run first (Step 2 missing case_step_features.parquet).')
    raise SystemExit(1)

features_df = pd.read_parquet(FEATURES_PARQUET)
loaded_from = FEATURES_PARQUET

features_df['timestamp'] = pd.to_datetime(features_df['timestamp'], utc=True, errors='coerce')

required_cols = [
    'step_index', 'trace_length', 'time_since_case_start_hours', 'time_since_prev_hours',
    'rework_count_activity', 'branch_label', 'branch_confidence',
    'is_terminal_event', 'case_completed', 'activity'
]
missing = [c for c in required_cols if c not in features_df.columns]
if missing:
    raise ValueError(f'Missing required columns in case_step_features: {missing}')

print(f'Loaded {len(features_df):,} rows from {loaded_from.name}')
features_df[['municipality', 'activity', 'branch_label', 'branch_confidence']].head(5)

Loaded 262,628 rows from case_step_features.parquet


,municipality,activity,branch_label,branch_confidence
0,1,register submission date request,unknown,0.0
1,1,phase application received,refusal,1.0
2,1,terminate on request,refusal,1.0
3,1,applicant is stakeholder,refusal,1.0
4,1,no permit needed or only notification needed,refusal,1.0


## 4.1 Action dictionary

The action set is manager-oriented for policy simulation (routing, staffing, sequencing, and compliance).

In [3]:
ACTIONS = [
    'assign_to_primary_team',
    'outsource_to_volunteer_pool',
    'rebalance_overloaded_queue',
    'merge_tasks_under_role',
    'prioritize_urgent_case',
    'defer_until_objections_resolved',
    'escalate_to_higher_authority',
    'skip_optional_subprocess',
    'add_temporary_staff',
    'adjust_staffing_by_case_volume',
    'enable_cross_trained_pool',
    'relax_rules_for_low_risk',
    'trigger_high_cost_escalation',
    'reroute_from_overloaded_employee',
    'close_case',
]

ACTION_TO_ID = {action: i for i, action in enumerate(ACTIONS)}
ID_TO_ACTION = {i: action for action, i in ACTION_TO_ID.items()}

rule_descriptions = {
    'assign_to_primary_team': 'Default manager action for active non-terminal cases.',
    'outsource_to_volunteer_pool': 'Valid when workload/volume pressure and delay are both high.',
    'rebalance_overloaded_queue': 'Valid when rework or queue-delay suggests local overload.',
    'merge_tasks_under_role': 'Valid when volume pressure is high and risk is not high.',
    'prioritize_urgent_case': 'Valid when case-age is high or risk branch indicates urgency.',
    'defer_until_objections_resolved': 'Valid when objection/appeal signals are present.',
    'escalate_to_higher_authority': 'Valid under extreme delay, suspension/refusal, or persistent rework.',
    'skip_optional_subprocess': 'Valid only on optional-step signals for low-risk cases.',
    'add_temporary_staff': 'Valid under simultaneous high volume and extreme case delay.',
    'adjust_staffing_by_case_volume': 'Valid when municipality-level case volume is elevated.',
    'enable_cross_trained_pool': 'Valid when volume or rework pressure is elevated.',
    'relax_rules_for_low_risk': 'Valid only for low-risk, high-confidence, non-objection contexts.',
    'trigger_high_cost_escalation': 'Valid on high-risk and high-delay proxy combinations.',
    'reroute_from_overloaded_employee': 'Valid when workload proxy is high (delay or rework).',
    'close_case': 'Valid only at terminal/completed states.',
}

print('Action mapping:')
for a in ACTIONS:
    print(f'{ACTION_TO_ID[a]} -> {a}')

Action mapping:
0 -> assign_to_primary_team
1 -> outsource_to_volunteer_pool
2 -> rebalance_overloaded_queue
3 -> merge_tasks_under_role
4 -> prioritize_urgent_case
5 -> defer_until_objections_resolved
6 -> escalate_to_higher_authority
7 -> skip_optional_subprocess
8 -> add_temporary_staff
9 -> adjust_staffing_by_case_volume
10 -> enable_cross_trained_pool
11 -> relax_rules_for_low_risk
12 -> trigger_high_cost_escalation
13 -> reroute_from_overloaded_employee
14 -> close_case


## 4.2 Deterministic action-mask rules

Rules are deterministic functions of state features only (no randomness).
Thresholds are data-driven quantiles from the current feature table.

In [4]:
delay_q75 = float(features_df['time_since_case_start_hours'].quantile(0.75))
delay_q90 = float(features_df['time_since_case_start_hours'].quantile(0.90))
prev_delay_q75 = float(features_df['time_since_prev_hours'].quantile(0.75))
rework_high = 2
rework_very_high = 3

MISSING_INFO_TERMS = [
    'missing', 'completeness', 'incomplete', 'supplement',
]
OBJECTION_TERMS = [
    'objection', 'appeal', 'complaint',
]
OPTIONAL_STEP_TERMS = [
    'advice', 'publication', 'draft', 'announce', 'intake review',
]

municipality_case_volume = (
    features_df[['municipality', 'case_id']].drop_duplicates().groupby('municipality').size()
)
volume_q75 = float(municipality_case_volume.quantile(0.75))

# Builds a dictionary of boolean flags for the given row that indicate various state conditions relevant for action masking
def build_state_flags(row: pd.Series):
    activity_text = str(row.get('activity', '')).lower()

    is_terminal = bool(row.get('is_terminal_event', False)) or bool(row.get('case_completed', False))
    high_case_delay = float(row.get('time_since_case_start_hours', 0.0)) >= delay_q75
    extreme_case_delay = float(row.get('time_since_case_start_hours', 0.0)) >= delay_q90
    high_step_delay = float(row.get('time_since_prev_hours', 0.0)) >= prev_delay_q75
    high_rework = int(row.get('rework_count_activity', 0)) >= rework_high
    very_high_rework = int(row.get('rework_count_activity', 0)) >= rework_very_high

    branch_label = str(row.get('branch_label', 'unknown'))
    branch_conf = float(row.get('branch_confidence', 0.0))
    low_branch_conf = branch_conf < 0.70

    is_refusal = branch_label == 'refusal'
    is_suspension = branch_label == 'suspension'
    is_completeness = branch_label == 'completeness'

    has_missing_signal = any(term in activity_text for term in MISSING_INFO_TERMS) or is_completeness
    has_objection_signal = any(term in activity_text for term in OBJECTION_TERMS)
    has_optional_signal = any(term in activity_text for term in OPTIONAL_STEP_TERMS)

    high_risk = bool(is_refusal or is_suspension or low_branch_conf)
    low_risk = bool((not high_risk) and (branch_conf >= 0.80) and (not has_objection_signal))

    m = row.get('municipality', np.nan)
    m = int(m) if pd.notna(m) else None
    case_volume = int(municipality_case_volume.get(m, 0)) if m is not None else 0
    high_volume = case_volume >= volume_q75

    overloaded_proxy = bool(high_step_delay or high_rework or extreme_case_delay)

    return {
        'is_terminal': is_terminal,
        'high_case_delay': high_case_delay,
        'extreme_case_delay': extreme_case_delay,
        'high_step_delay': high_step_delay,
        'high_rework': high_rework,
        'very_high_rework': very_high_rework,
        'high_risk': high_risk,
        'low_risk': low_risk,
        'has_missing_signal': has_missing_signal,
        'has_objection_signal': has_objection_signal,
        'has_optional_signal': has_optional_signal,
        'high_volume': high_volume,
        'overloaded_proxy': overloaded_proxy,
        'is_refusal': is_refusal,
        'is_suspension': is_suspension,
    }

# Given a row of features, returns a dictionary mapping each action to a boolean indicating whether that action is valid for the state
def action_mask_from_state(row: pd.Series):
    flags = build_state_flags(row)

    mask = {action: False for action in ACTIONS}

    if flags['is_terminal']:
        mask['close_case'] = True
        return mask

    mask['assign_to_primary_team'] = True
    mask['outsource_to_volunteer_pool'] = bool(flags['high_volume'] and flags['high_case_delay'])
    mask['rebalance_overloaded_queue'] = bool(flags['overloaded_proxy'])
    mask['merge_tasks_under_role'] = bool(flags['high_volume'] and not flags['high_risk'])

    mask['prioritize_urgent_case'] = bool(flags['high_case_delay'] or flags['high_risk'])
    mask['defer_until_objections_resolved'] = bool(flags['has_objection_signal'])
    mask['escalate_to_higher_authority'] = bool(flags['extreme_case_delay'] or flags['very_high_rework'] or flags['is_suspension'] or flags['is_refusal'])
    mask['skip_optional_subprocess'] = bool(flags['has_optional_signal'] and flags['low_risk'])

    mask['add_temporary_staff'] = bool(flags['high_volume'] and flags['extreme_case_delay'])
    mask['adjust_staffing_by_case_volume'] = bool(flags['high_volume'])
    mask['enable_cross_trained_pool'] = bool(flags['high_volume'] or flags['high_rework'])

    mask['relax_rules_for_low_risk'] = bool(flags['low_risk'])
    mask['trigger_high_cost_escalation'] = bool(flags['high_risk'] and flags['high_case_delay'])
    mask['reroute_from_overloaded_employee'] = bool(flags['overloaded_proxy'])

    mask['close_case'] = False
    return mask

print('Thresholds:')
print('delay_q75:', round(delay_q75, 4))
print('delay_q90:', round(delay_q90, 4))
print('prev_delay_q75:', round(prev_delay_q75, 4))
print('rework_high:', rework_high)
print('rework_very_high:', rework_very_high)
print('volume_q75:', round(volume_q75, 4))

Thresholds:
delay_q75: 1455.1312
delay_q90: 3033.9894
prev_delay_q75: 0.0019
rework_high: 2
rework_very_high: 3
volume_q75: 1199.0


In [5]:
masks = features_df.apply(action_mask_from_state, axis=1)
mask_df = pd.DataFrame(masks.tolist())

for action in ACTIONS:
    features_df[f'valid__{action}'] = mask_df[action].astype(bool)

features_df['num_valid_actions'] = mask_df.sum(axis=1)

action_validity_summary = pd.DataFrame({
    'action': ACTIONS,
    'action_id': [ACTION_TO_ID[a] for a in ACTIONS],
    'valid_count': [int(mask_df[a].sum()) for a in ACTIONS],
    'valid_rate': [float(mask_df[a].mean()) for a in ACTIONS],
    'rule_description': [rule_descriptions[a] for a in ACTIONS],
}).sort_values('action_id').reset_index(drop=True)

action_validity_summary

,action,action_id,valid_count,valid_rate,rule_description
0,assign_to_primary_team,0,256979,0.978490,Default manager action for active non-terminal...
1,outsource_to_volunteer_pool,1,25496,0.097080,Valid when workload/volume pressure and delay ...
2,rebalance_overloaded_queue,2,82591,0.314479,Valid when rework or queue-delay suggests loca...
3,merge_tasks_under_role,3,39187,0.149211,Valid when volume pressure is high and risk is...
4,prioritize_urgent_case,4,184402,0.702141,Valid when case-age is high or risk branch ind...
5,defer_until_objections_resolved,5,5411,0.020603,Valid when objection/appeal signals are present.
6,escalate_to_higher_authority,6,78815,0.300101,"Valid under extreme delay, suspension/refusal,..."
7,skip_optional_subprocess,7,3079,0.011724,Valid only on optional-step signals for low-ri...
8,add_temporary_staff,8,9655,0.036763,Valid under simultaneous high volume and extre...
9,adjust_staffing_by_case_volume,9,109290,0.416140,Valid when municipality-level case volume is e...


In [6]:
# Checks

# 1) Every state has at least one valid action
states_with_zero_valid_actions = int((features_df['num_valid_actions'] == 0).sum())

# 2) Determinism: same feature signature -> same mask signature
state_key_cols = [
    'municipality',
    'step_index', 'trace_length', 'time_since_case_start_hours', 'time_since_prev_hours',
    'rework_count_activity', 'branch_label', 'branch_confidence',
    'is_terminal_event', 'case_completed', 'activity'
]

check_df = features_df[state_key_cols + [f'valid__{a}' for a in ACTIONS]].copy()

# Round continuous values to avoid floating precision artifacts in grouping
check_df['time_since_case_start_hours'] = check_df['time_since_case_start_hours'].round(6)
check_df['time_since_prev_hours'] = check_df['time_since_prev_hours'].round(6)
check_df['branch_confidence'] = check_df['branch_confidence'].round(6)

mask_signature_cols = [f'valid__{a}' for a in ACTIONS]
sig_counts = (
    check_df.groupby(state_key_cols, dropna=False)[mask_signature_cols]
    .nunique()
    .max(axis=1)
)
nondeterministic_signatures = int((sig_counts > 1).sum())

print('States with zero valid actions:', states_with_zero_valid_actions)
print('Non-deterministic signatures:', nondeterministic_signatures)
print('Check 1 passed:', states_with_zero_valid_actions == 0)
print('Check 2 passed:', nondeterministic_signatures == 0)

States with zero valid actions: 0
Non-deterministic signatures: 0
Check 1 passed: True
Check 2 passed: True


In [7]:
export_rows = []
for action in ACTIONS:
    export_rows.append({
        'action_id': ACTION_TO_ID[action],
        'action_name': action,
        'rule_description': rule_descriptions[action],
        'delay_q75': delay_q75,
        'delay_q90': delay_q90,
        'prev_delay_q75': prev_delay_q75,
        'rework_high': rework_high,
        'rework_very_high': rework_very_high,
        'volume_q75': volume_q75,
        'valid_count': int(mask_df[action].sum()),
        'valid_rate': float(mask_df[action].mean()),
    })

valid_action_space_df = pd.DataFrame(export_rows).sort_values('action_id').reset_index(drop=True)
valid_action_space_df.to_csv(ACTION_SPACE_PATH, index=False)
print('Saved:', ACTION_SPACE_PATH.resolve())
valid_action_space_df

Saved: /home/praharsha/Desktop/bueracratic-workflow-analyzer/output/valid_action_space.csv


,action_id,action_name,rule_description,delay_q75,delay_q90,prev_delay_q75,rework_high,rework_very_high,volume_q75,valid_count,valid_rate
0,0,assign_to_primary_team,Default manager action for active non-terminal...,1455.131181,3033.989444,0.001944,2,3,1199.0,256979,0.978490
1,1,outsource_to_volunteer_pool,Valid when workload/volume pressure and delay ...,1455.131181,3033.989444,0.001944,2,3,1199.0,25496,0.097080
2,2,rebalance_overloaded_queue,Valid when rework or queue-delay suggests loca...,1455.131181,3033.989444,0.001944,2,3,1199.0,82591,0.314479
3,3,merge_tasks_under_role,Valid when volume pressure is high and risk is...,1455.131181,3033.989444,0.001944,2,3,1199.0,39187,0.149211
4,4,prioritize_urgent_case,Valid when case-age is high or risk branch ind...,1455.131181,3033.989444,0.001944,2,3,1199.0,184402,0.702141
5,5,defer_until_objections_resolved,Valid when objection/appeal signals are present.,1455.131181,3033.989444,0.001944,2,3,1199.0,5411,0.020603
6,6,escalate_to_higher_authority,"Valid under extreme delay, suspension/refusal,...",1455.131181,3033.989444,0.001944,2,3,1199.0,78815,0.300101
7,7,skip_optional_subprocess,Valid only on optional-step signals for low-ri...,1455.131181,3033.989444,0.001944,2,3,1199.0,3079,0.011724
8,8,add_temporary_staff,Valid under simultaneous high volume and extre...,1455.131181,3033.989444,0.001944,2,3,1199.0,9655,0.036763
9,9,adjust_staffing_by_case_volume,Valid when municipality-level case volume is e...,1455.131181,3033.989444,0.001944,2,3,1199.0,109290,0.416140


## Step 4 complete

You now have a manager-oriented action dictionary and deterministic validity rules exported to `./output/valid_action_space.csv`.

Next steps:
- Step 5 baseline reward function,
- Step 5.5 KPI-driven reward optimization.